In [1]:
import pandas as pd


In [3]:

df = pd.read_csv('/home/milijanas/projects/genomic-rag-assistant/data/uniprotkb_Human_proteins_AND_model_orga_2026_03_20.tsv', sep='\t')


In [5]:

df.head(5)

,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Pathway,Function [CC],Sequence,Subcellular location [CC],Gene Ontology (biological process),Gene Ontology (molecular function),Involvement in disease,Keywords,Interacts with
0,A0A0B4J2F0,reviewed,PIOS1_HUMAN,Protein PIGBOS1 (PIGB opposite strand protein 1),PIGBOS1,Homo sapiens (Human),54,NaN,FUNCTION: Plays a role in regulation of the un...,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,SUBCELLULAR LOCATION: Mitochondrion outer memb...,regulation of endoplasmic reticulum unfolded p...,NaN,NaN,Direct protein sequencing;Membrane;Mitochondri...,Q96S66
1,A0A0U1RRE5,reviewed,NBDY_HUMAN,Negative regulator of P-body association (P-bo...,NBDY LINC01420,Homo sapiens (Human),68,NaN,FUNCTION: Promotes dispersal of P-body compone...,MGDQPCASGRSTLPPGNAREAKPPKKRCLLAPRWDYPEGTPNGGST...,"SUBCELLULAR LOCATION: Cytoplasm, P-body {ECO:0...",mRNA processing [GO:0006397]; negative regulat...,NaN,NaN,Cytoplasm;mRNA processing;Proteomics identific...,Q9NPI6; Q6P2E9
2,A0A8I5KQE6,reviewed,RPSA2_HUMAN,Small ribosomal subunit protein uS2B (37 kDa l...,RPSA2 RPSA RPSAP58,Homo sapiens (Human),295,NaN,FUNCTION: Required for the assembly and/or sta...,MSGALDVLQMKEEDVLKFLAAGTHLGGTNLDFQMEHYIYKRKSDGI...,SUBCELLULAR LOCATION: Cell membrane {ECO:00002...,cytoplasmic translation [GO:0002181]; ribosoma...,laminin binding [GO:0043236]; laminin receptor...,NaN,Acetylation;Cell membrane;Cytoplasm;Membrane;N...,NaN
3,A0AV96,reviewed,RBM47_HUMAN,RNA-binding protein 47 (RNA-binding motif prot...,RBM47,Homo sapiens (Human),593,NaN,FUNCTION: Single-stranded RNA-binding protein ...,MTAEDSTAAMSSDSAAGSSAKVPEGVAGAPNEAALLALMERTGYSM...,SUBCELLULAR LOCATION: Nucleus {ECO:0000269|Pub...,3'-UTR-mediated mRNA stabilization [GO:0070935...,enzyme binding [GO:0019899]; enzyme-substrate ...,NaN,3D-structure;Alternative splicing;Cytoplasm;Me...,Q5SWW7; Q6P1W5; Q9UQM7; Q8N6W0; B7ZLH0; P32243...
4,A0AVF1,reviewed,IFT56_HUMAN,Intraflagellar transport protein 56 (Tetratric...,IFT56 TTC26,Homo sapiens (Human),554,NaN,FUNCTION: Component of the intraflagellar tran...,MMLSRAKPAVGRGVQHTDKRKKKGRKIPKLEELLSKRDFTGAITLL...,"SUBCELLULAR LOCATION: Cell projection, cilium ...",axoneme assembly [GO:0035082]; cilium assembly...,intraciliary transport particle B binding [GO:...,"DISEASE: Biliary, renal, neurologic, and skele...",Alternative splicing;Cell projection;Ciliopath...,Q9UG01; Q9H7X7; Q9Y547; Q9BW83; Q9NQC8; Q9Y366...


In [7]:
df.columns

Index(['Entry', 'Reviewed', 'Entry Name', 'Protein names', 'Gene Names',
       'Organism', 'Length', 'Pathway', 'Function [CC]', 'Sequence',
       'Subcellular location [CC]', 'Gene Ontology (biological process)',
       'Gene Ontology (molecular function)', 'Involvement in disease',
       'Keywords', 'Interacts with'],
      dtype='object')

In [14]:
docs = []
cols = ['Entry', 'Entry Name', 'Protein names', 'Gene Names', 'Length', 'Function [CC]',
       'Subcellular location [CC]', 'Gene Ontology (biological process)',
       'Gene Ontology (molecular function)', 'Involvement in disease',
       'Keywords', 'Interacts with']

for _, row in df.iterrows():
    string_fields = [f"{col}:{row[col]}" for col in df.columns]
    row_string = '\n'.join(string_fields)
    docs.append(row_string)



In [15]:
print(docs[:10])

['Entry:A0A0B4J2F0\nReviewed:reviewed\nEntry Name:PIOS1_HUMAN\nProtein names:Protein PIGBOS1 (PIGB opposite strand protein 1)\nGene Names:PIGBOS1\nOrganism:Homo sapiens (Human)\nLength:54\nPathway:nan\nFunction [CC]:FUNCTION: Plays a role in regulation of the unfolded protein response triggered by endoplasmic reticulum (ER) stress resulting from the presence of unfolded proteins in the ER lumen. {ECO:0000269|PubMed:31653868}.\nSequence:MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLVQESEEKKS\nSubcellular location [CC]:SUBCELLULAR LOCATION: Mitochondrion outer membrane {ECO:0000269|PubMed:31653868}; Single-pass membrane protein {ECO:0000255}.\nGene Ontology (biological process):regulation of endoplasmic reticulum unfolded protein response [GO:1900101]; response to unfolded protein [GO:0006986]\nGene Ontology (molecular function):nan\nInvolvement in disease:nan\nKeywords:Direct protein sequencing;Membrane;Mitochondrion;Mitochondrion outer membrane;Proteomics identification;Reference proteo

In [16]:
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions

# Initialize a ChromaDB client (using in-memory persisting for demo; adapt as needed)
client = chromadb.Client(Settings(
    persist_directory="./chromadb_rag_store",  # you can change the directory as needed
    chroma_db_impl="duckdb+parquet"
))

# Set up an embedding function (e.g., OpenAI). You'll need an actual API key for production use.
# Replace embedding_functions.DefaultEmbeddingFunction() with your custom embedding if needed.
embedding_function = embedding_functions.DefaultEmbeddingFunction()

collection = client.get_or_create_collection(
    name="uniprot_docs",
    embedding_function=embedding_function
)

# Add docs to the collection
collection.add(
    documents=docs,
    ids=[f"doc_{i}" for i in range(len(docs))]
)

# Example: Retrieval for RAG (retrieve top 3 relevant docs for a sample query)
query = "mitochondrial membrane proteins implicated in disease"
results = collection.query(
    query_texts=[query],
    n_results=3
)

for i, doc in enumerate(results['documents'][0]):
    print(f"\nResult {i+1}:\n{doc}")


ValueError: [91mYou are using a deprecated configuration of Chroma.

[94mIf you do not have data you wish to migrate, you only need to change how you construct
your Chroma client. Please see the "New Clients" section of https://docs.trychroma.com/deployment/migration.
________________________________________________________________________________________________

If you do have data you wish to migrate, we have a migration tool you can use in order to
migrate your data to the new Chroma architecture.
Please `pip install chroma-migrate` and run `chroma-migrate` to migrate your data and then
change how you construct your Chroma client.

See https://docs.trychroma.com/deployment/migration for more information or join our discord at https://discord.gg/MMeYNTmh3x for help![0m